In [4]:
import os
import sqlite3

# Create necesary folders
os.makedirs("templates", exist_ok=True)
os.makedirs("static/css", exist_ok=True)
os.makedirs("static/js", exist_ok=True)

# 1. config.py
with open("config.py", "w") as f:
    f.write('''
DB_NAME = "sensor_history.db"
LOG_FILE = "app_activity.log"
SAMPLING_INTERVAL = 5
STALE_THRESHOLD_SECONDS = 15
TEMP_HIGH_THRESHOLD = 75.0
ALERT_PERSISTENCE_COUNT = 3
ALERT_COOLDOWN_SECONDS = 30
''')

# 2. database.py
with open("database.py", "w") as f:
    f.write('''
import sqlite3
import datetime
from config import DB_NAME

def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS sensor_readings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
            temperature REAL,
            humidity REAL,
            status TEXT CHECK(status IN ('ACCEPTED', 'REJECTED', 'MISSING', 'STALE'))
        )
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS system_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
            event_type TEXT,
            message TEXT,
            health_state TEXT
        )
    """)
    conn.commit()
    conn.close()

def log_event(event_type, message, health_state="HEALTHY"):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("INSERT INTO system_logs (event_type, message, health_state) VALUES (?, ?, ?)",
                   (event_type, message, health_state))
    conn.commit()
    conn.close()

def insert_reading(temp, humidity, status):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    now = datetime.datetime.utcnow().isoformat()
    cursor.execute("INSERT INTO sensor_readings (timestamp, temperature, humidity, status) VALUES (?, ?, ?, ?)",
                   (now, temp, humidity, status))
    conn.commit()
    conn.close()

def get_latest_reading():
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM sensor_readings ORDER BY id DESC LIMIT 1")
    row = cursor.fetchone()
    conn.close()
    return dict(row) if row else None

def get_historical_readings(limit=50):
    conn = sqlite3.connect(DB_NAME)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM sensor_readings ORDER BY id DESC LIMIT ?", (limit,))
    rows = cursor.fetchall()
    conn.close()
    return [dict(r) for r in rows]
''')

# 3. sensor_generator.py
with open("sensor_generator.py", "w") as f:
    f.write('''
import time
import random
from database import insert_reading, init_db
from config import SAMPLING_INTERVAL

def run_sensor_generator(simulated_mode="NORMAL"):
    init_db()
    while True:
        if simulated_mode == "STALE":
            time.sleep(SAMPLING_INTERVAL)
            continue
        if simulated_mode == "HIGH_TEMP":
            temperature = round(random.uniform(76.0, 85.0), 2)
            humidity = round(random.uniform(30.0, 40.0), 2)
        else:
            temperature = round(random.uniform(20.0, 25.0), 2)
            humidity = round(random.uniform(45.0, 55.0), 2)
        insert_reading(temperature, humidity, "ACCEPTED")
        time.sleep(SAMPLING_INTERVAL)

if __name__ == "__main__":
    run_sensor_generator(simulated_mode="NORMAL")
''')

# 4. monitor_engine.py
with open("monitor_engine.py", "w") as f:
    f.write('''
import time
import logging
from datetime import datetime
from config import LOG_FILE, STALE_THRESHOLD_SECONDS, TEMP_HIGH_THRESHOLD, ALERT_PERSISTENCE_COUNT, ALERT_COOLDOWN_SECONDS
from database import get_latest_reading, log_event, insert_reading

logging.basicConfig(filename=LOG_FILE, level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

class MonitorEngine:
    def __init__(self):
        self.health_state = "RECOVERING"
        self.consecutive_high_temp = 0
        self.last_alert_time = 0
        self.alert_active = False
        log_event("STARTUP", "System startup initiated.", self.health_state)
        time.sleep(2)
        self.health_state = "HEALTHY"
        log_event("STATE_CHANGE", "System transitioned to HEALTHY state.", self.health_state)

    def evaluate(self):
        latest = get_latest_reading()
        now = datetime.utcnow()
        if not latest:
            self.health_state = "DEGRADED"
            return
        last_ts = datetime.fromisoformat(latest['timestamp'])
        time_diff = (now - last_ts).total_seconds()
        if time_diff > STALE_THRESHOLD_SECONDS:
            if self.health_state != "DEGRADED":
                self.health_state = "DEGRADED"
                log_event("STALE_DATA", f"Sensor stale ({time_diff:.1f}s delay)", self.health_state)
                insert_reading(latest['temperature'], latest['humidity'], 'STALE')
            return
        if self.health_state == "DEGRADED" and time_diff <= STALE_THRESHOLD_SECONDS:
            self.health_state = "HEALTHY"
            log_event("RECOVERY", "Sensor resumed.", self.health_state)
        temp = latest['temperature']
        if temp >= TEMP_HIGH_THRESHOLD:
            self.consecutive_high_temp += 1
            if self.consecutive_high_temp >= ALERT_PERSISTENCE_COUNT:
                current_time = time.time()
                if (current_time - self.last_alert_time) > ALERT_COOLDOWN_SECONDS:
                    self.alert_active = True
                    self.health_state = "DEGRADED"
                    self.last_alert_time = current_time
                    log_event("ALERT_ACTIVE", f"HIGH TEMP: {temp}°C", self.health_state)
        else:
            if self.alert_active:
                self.alert_active = False
                log_event("ALERT_CLEARED", f"Temp normalized: {temp}°C", self.health_state)
            self.consecutive_high_temp = 0

if __name__ == "__main__":
    engine = MonitorEngine()
    while True:
        engine.evaluate()
        time.sleep(3)
''')

# 5. app.py
with open("app.py", "w") as f:
    f.write('''
from flask import Flask, render_template_string, jsonify
from database import init_db, get_latest_reading, get_historical_readings
import sqlite3
from config import DB_NAME, STALE_THRESHOLD_SECONDS, TEMP_HIGH_THRESHOLD

app = Flask(__name__)
init_db()

HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Unattended System Monitor</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
</head>
<body class="bg-light">
<div class="container py-4">
    <h1 class="mb-4">Unattended System Monitoring Dashboard</h1>
    <div class="row mb-4">
        <div class="col-md-3">
            <div class="card text-center p-3 shadow-sm">
                <h5 class="text-muted">System Health</h5>
                <h2 id="health-state" class="fw-bold">LOADING...</h2>
            </div>
        </div>
        <div class="col-md-3">
            <div class="card text-center p-3 shadow-sm">
                <h5 class="text-muted">Current Temp</h5>
                <h2 id="current-temp" class="fw-bold">-- °C</h2>
            </div>
        </div>
        <div class="col-md-3">
            <div class="card text-center p-3 shadow-sm">
                <h5 class="text-muted">Sensor Status</h5>
                <h2 id="sensor-status" class="fw-bold">--</h2>
            </div>
        </div>
        <div class="col-md-3">
            <div class="card text-center p-3 shadow-sm">
                <h5 class="text-muted">Latest Update</h5>
                <p id="latest-timestamp" class="small text-muted mt-2">--</p>
            </div>
        </div>
    </div>
    <div class="card p-3 shadow-sm">
        <h5>Historical Temperature Readings</h5>
        <canvas id="historyChart" height="100"></canvas>
    </div>
</div>
<script>
    let chart;
    function initChart() {
        const ctx = document.getElementById('historyChart').getContext('2d');
        chart = new Chart(ctx, {
            type: 'line',
            data: { labels: [], datasets: [{ label: 'Temperature (°C)', data: [], borderColor: '#0d6efd', fill: false }] },
            options: { responsive: true, scales: { y: { suggestedMin: 15, suggestedMax: 90 } } }
        });
    }
    async function update() {
        try {
            const res = await fetch('/api/status');
            const data = await res.json();
            if (data.current_reading) {
                document.getElementById('current-temp').innerText = `${data.current_reading.temperature} °C`;
                document.getElementById('sensor-status').innerText = data.current_reading.status;
                document.getElementById('latest-timestamp').innerText = data.current_reading.timestamp;
            }
            const healthElem = document.getElementById('health-state');
            healthElem.innerText = data.health_state;
            healthElem.className = `fw-bold ${data.health_state === 'HEALTHY' ? 'text-success' : 'text-danger'}`;

            const histRes = await fetch('/api/history');
            const histData = await histRes.json();
            if (chart && Array.isArray(histData)) {
                const reversed = histData.reverse();
                chart.data.labels = reversed.map(r => r.timestamp.split('T')[1]?.substring(0,8) || r.timestamp);
                chart.data.datasets[0].data = reversed.map(r => r.temperature);
                chart.update();
            }
        } catch (e) {}
    }
    document.addEventListener('DOMContentLoaded', () => {
        initChart();
        update();
        setInterval(update, 3000);
    });
</script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

@app.route('/api/status')
def api_status():
    latest = get_latest_reading()
    health_state = "HEALTHY"
    try:
        conn = sqlite3.connect(DB_NAME)
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM system_logs ORDER BY id DESC LIMIT 1')
        last_log = cursor.fetchone()
        conn.close()
        if last_log:
            health_state = last_log['health_state']
    except Exception:
        pass

    return jsonify({
        "current_reading": latest,
        "health_state": health_state,
        "stale_threshold": STALE_THRESHOLD_SECONDS,
        "alert_threshold": TEMP_HIGH_THRESHOLD
    })

@app.route('/api/history')
def api_history():
    return jsonify(get_historical_readings(limit=30))

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
''')

# 6. supervisord.conf
with open("supervisord.conf", "w") as f:
    f.write('''
[supervisord]
nodaemon=true
user=root

[program:flask_app]
command=python3 app.py
autorestart=true

[program:monitor_engine]
command=python3 monitor_engine.py
autorestart=true

[program:sensor_generator]
command=python3 sensor_generator.py
autorestart=true
''')

# Initialize BD
from database import init_db
init_db()

print("¡Files corrected and SQLite initialization completed successfully!")

¡Files corrected and SQLite initialization completed successfully!


In [5]:
%%bash -b
# Stop only previous instances of our scripts
pkill -f "python3 app.py" || true
pkill -f "python3 monitor_engine.py" || true
pkill -f "python3 sensor_generator.py" || true

# Launch the 3 components in the background
nohup python3 app.py > flask.log 2>&1 &
nohup python3 monitor_engine.py > monitor.log 2>&1 &
nohup python3 sensor_generator.py > sensor.log 2>&1 &

In [6]:
import time
from google.colab import output

time.sleep(2) # Wait for Flask to finish starting up on port 5000
print("=" * 60)
print("🔗 YOUR DASHBOARD IS READY:")
print(output.eval_js("google.colab.kernel.proxyPort(5000)"))
print("=" * 60)

🔗 YOUR DASHBOARD IS READY:
https://5000-m-s-kkb-use1d0-3mnubb06447px-d.us-east1-0.prod.colab.dev


In [7]:
%%bash
# Switch sensor simulator to high temperature mode (>75 °C)
sed -i 's/simulated_mode="NORMAL"/simulated_mode="HIGH_TEMP"/g' sensor_generator.py
pkill -f "python3 sensor_generator.py" || true
nohup python3 sensor_generator.py > sensor.log 2>&1 &

In [8]:
%%bash
# Restore sensor simulator to normal mode
sed -i 's/simulated_mode="HIGH_TEMP"/simulated_mode="NORMAL"/g' sensor_generator.py
pkill -f "python3 sensor_generator.py" || true
nohup python3 sensor_generator.py > sensor.log 2>&1 &

In [9]:
%%bash
# Stop sensor data transmission
pkill -f "python3 sensor_generator.py" || true

In [10]:
%%bash
# Resume sensor execution
nohup python3 sensor_generator.py > sensor.log 2>&1 &

In [13]:
import sqlite3
import pandas as pd

# Query system logs directly from SQLite
conn = sqlite3.connect("sensor_history.db")
df = pd.read_sql_query("SELECT id, timestamp, event_type, message, health_state FROM system_logs ORDER BY id DESC LIMIT 20", conn)
conn.close()

# Display the formatted table on the screen
print("=== SYSTEM LOGS & EVENT HISTORY ===")
display(df)

=== SYSTEM LOGS & EVENT HISTORY ===


,id,timestamp,event_type,message,health_state
0,21,2026-10-18 21:57:27,RECOVERY,Sensor resumed.,HEALTHY
1,20,2026-10-18 21:57:24,STALE_DATA,Sensor stale (15.0s delay),DEGRADED
2,19,2026-10-18 21:57:12,RECOVERY,Sensor resumed.,HEALTHY
3,18,2026-10-18 21:57:09,STALE_DATA,Sensor stale (15.0s delay),DEGRADED
4,17,2026-10-18 21:56:57,RECOVERY,Sensor resumed.,HEALTHY
5,16,2026-10-18 21:56:54,STALE_DATA,Sensor stale (15.0s delay),DEGRADED
6,15,2026-10-18 21:56:42,RECOVERY,Sensor resumed.,HEALTHY
7,14,2026-10-18 21:56:39,STALE_DATA,Sensor stale (15.0s delay),DEGRADED
8,13,2026-10-18 21:56:27,RECOVERY,Sensor resumed.,HEALTHY
9,12,2026-10-18 21:56:24,STALE_DATA,Sensor stale (15.0s delay),DEGRADED
